In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

import time
import torch
from torch import nn, Tensor
from torch.nn import functional as F

import math
from math import ceil, log2

--2025-01-02 17:59:50--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.01s   

2025-01-02 17:59:51 (109 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 512 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Logarithmic Memory

In [3]:
class Summarizer(nn.Module):
    def __init__(self, embedding: int, bank: int = 1):
        super(Summarizer, self).__init__()
        self.weight = nn.Parameter(torch.randn(bank, 2 * embedding, embedding))
        self.bias = nn.Parameter(torch.randn(1, bank, 1, embedding))
        nn.init.xavier_normal_(self.weight)
        nn.init.xavier_normal_(self.bias)
        self.weight.data = self.weight.data * 0.1
        self.bias.data = self.bias.data * 0.1
        # self.layer = nn.Conv1d(2*embedding*bank, embedding*bank, 1)

    def forward(self, x: Tensor):
        # batch, bank, seq, embed = x.shape
        # x = x.transpose(-1, -2)
        # x = x.reshape(batch, embed*bank, seq)
        # x = self.layer(x)
        # x = x.view(batch, bank, (embed//2), seq)
        # x = x.transpose(-1, -2)
        # return x

        return torch.einsum("b n s d, n d e -> b n s e", x, self.weight) + self.bias


class LogarithmicMemory(nn.Module):
    """
    A base module for exploring memory access with a logarithmic addressing scheme.

    This module provides a framework for experimenting with different methods
    of interacting with a memory structure. It defines a logarithmic memory
    pointer and several methods to access this memory, intended for research
    and experimentation purposes. It offers parallel, sequential, and attention-based
    methods of accessing the memory.
    """
    def __init__(self, embedding: int, heads: int = 1, bank: int = 1, parallelize=True):
        """
        Initializes the LogarithmicMemory module.
        """
        super().__init__()
        assert embedding % heads == 0, "Embedding dimension must be divisible by the number of attention heads."

        self.embedding = embedding
        self.heads = heads
        self.head_dim = embedding//heads
        self.scale = math.sqrt(self.head_dim)
        self.bank = bank
        self.parallelize = parallelize

        self.norm1 = nn.RMSNorm(embedding)
        self.norm2 = nn.RMSNorm(embedding)
        self.summarizer = Summarizer(embedding, bank)
        self.qkv = nn.Linear(embedding, 3*embedding)
        self.out_proj = nn.Linear(embedding, embedding)
        self.feedforward = nn.Sequential(
            nn.Linear(embedding, 4*embedding),
            nn.SiLU(),
            nn.Linear(4*embedding, embedding),
        )
        self.reset()

    def reset(self):
        """
        Resets the internal memory and the memory pointer.

        This function sets the internal memory (`self.memory`) to `None`,
        and the memory pointer (`self.ptr`) to 0. This allows for starting from
        a clean slate in each new experiment.
        """
        self.memory = None
        self.ptr = 0

    def summarize_table(self) -> list:
        """
        Summarizes a memory table based on a logarithmic addressing scheme.

        This function generates a logarithmic address by creating a sequence of
        binary differences between the current memory pointer (`self.ptr`) and
        the next one (`self.ptr + 1`). The memory pointer will also be incremented
        in the process.

        Returns:
            list: A list of boolean values representing the bit-wise differences between
                  consecutive memory pointers.
        """
        a, b = self.ptr, self.ptr+1
        self.ptr = b
        max_len = max(a.bit_length(), b.bit_length())
        return [(a ^ b) & (1 << i) != 0 for i in range(1, max_len)]

    def parallel(self, x: Tensor) -> Tensor: #[batch, seq, embedding]
        """
        Processes input in parallel based on the logarithmic memory addresses.

        This method is a placeholder for a parallel memory access scheme.
        It's intended to define how an input `x` is processed when interacting
        with the memory in a parallel fashion, using logarithmic addresses.

        Args:
            x (Tensor): Input tensor of shape (batch_size, seq_len, embedding_dim).

        Returns:
           Tensor: Output tensor of shape (batch_size, seq_len, 1+floor(log(seq_len))*bank, embedding_dim).
        """
        # Graph
        x_original = x
        x = x.unsqueeze(1).repeat(1, self.bank, 1, 1)
        batch_x, bank_x, seq_x, embed_x = x.shape

        x_last = x
        x_t = [x]
        for i in range(1, seq_x.bit_length()):
            x_temp = x_last[:, :, :-1, :] if x_last.size(-2) % 2 == 1 else x_last # set length to be even
            batch, bank, seq, embedding = x_temp.shape
            x_temp = x_temp.view(batch, bank, seq//2, 2*embedding) # concatenate each two tokens #[batch, bank, seq/2, embedding*2]
            x_last = self.summarizer(x_temp) # calculate the tree graph #[batch, bank, seq/2, embedding]
            x_temp = x_last.unsqueeze(-2).repeat(1, 1, 1, 2**i, 1).view(batch, bank, seq*2**(i-1), embedding) # sync shape of sequence by repeating #[batch, bank, seq_x, embedding]
            exceed_shape = -(2**i-1)+(seq_x-seq*2**(i-1)) # handle extra non-power 2 index
            exceed_shape = seq_x + 1 if exceed_shape == 0 else exceed_shape
            x_temp = torch.cat([torch.zeros(batch_x, bank_x, 2**i-1, embed_x, device=x.device), x_temp[:, :, :exceed_shape, :]], dim=-2) # add zeros triangle #[batch, bank_x, seq_x, embedding]
            x_t.append(x_temp)

        if len(x_t) == 1:
            return x_original.unsqueeze(-2)

        x = torch.stack(x_t[1:], dim=-2) # [batch, bank, seq, log(seq)-1, embedding]
        x = x.permute(0, 2, 3, 1, 4).contiguous() # [batch, seq, log(seq)-1, bank, embedding]
        x = x.view(batch_x, seq_x, -1, embed_x) # [batch, seq, log(seq)*bank-1, embedding]
        x = torch.cat([x_original.unsqueeze(-2), x], dim=-2) # [batch, seq, log(seq)*bank, embedding]
        return x

    def sequential(self, x: Tensor) -> Tensor: # [batch, seq, embedding]
        """
        Processes input sequentially, writing to and reading from the memory based on
        logarithmic addressing.

        This method is a placeholder for a sequential memory access scheme.
        It defines how an input `x` interacts with the memory one step at a time,
        using logarithmic addresses.

        Args:
            x (Tensor): Input tensor of shape (batch_size, 1, embedding_dim).

        Returns:
           Tensor: Output tensor of shape (batch_size, 1, 1+ceil(log(seq_len))*bank, embedding_dim).
        """
        # Memory
        x_original = x
        x = x.unsqueeze(1).repeat(1, self.bank, 1, 1) # [batch, bank, seq, embedding]
        batch_x, bank_x, seq_x, embed_x = x.shape

        new_memory = [x]
        for i, summarize in enumerate(self.summarize_table()):
            # summarize
            if summarize:
                combination = self.summarizer(torch.cat([self.memory[i], new_memory[i]], dim=-1))
                new_memory.append(combination)

            # copy
            else:
                new_memory.append(self.memory[i + 1])
        self.memory = new_memory

        if len(new_memory) == 1:
            return x_original.unsqueeze(-2)

        x = torch.stack(new_memory[1:], dim=-2) # [batch, bank, seq, log(seq)-1, embedding]
        x = x.permute(0, 2, 3, 1, 4).contiguous() # [batch, seq, log(seq)-1, bank, embedding]
        x = x.view(batch_x, seq_x, -1, embed_x) # [batch, seq, log(seq)*bank-1, embedding]
        x = torch.cat([x_original.unsqueeze(-2), x], dim=-2) # [batch, seq, log(seq)*bank, embedding]
        return x

    def attention(self, x: Tensor) -> Tensor: # [batch, seq, log(seq), embedding]
        """
        Processes input using an attention mechanism to read from the logarithmic memory.

        This method is a placeholder for implementing an attention-based memory access
        scheme. It describes the intended interaction of an input `x` with the
        memory, using the logarithmic address in an attention mechanism.

        Args:
            x (Tensor): Input tensor of shape (batch_size, seq_len or 1, 1+ceil(log(seq_len))*bank, embedding_dim).

        Returns:
           Tensor: Output tensor of shape (batch_size, seq_len or 1, embedding_dim).
        """
        # Attention
        batch_size, seq, log_seq, embedding = x.shape

        Q, K, V = torch.split(self.qkv(x) * (torch.cat([x]*3, dim=-1) != 0), [self.embedding]*3, dim=-1) # mask zeros values to remain the same

        Q = Q.view(batch_size, seq, log_seq, self.heads, self.head_dim).transpose(-2, -3)
        K = K[:, :, 0].view(batch_size, seq, 1, self.heads, self.head_dim).transpose(-2, -3)
        V = V.view(batch_size, seq, log_seq, self.heads, self.head_dim).transpose(-2, -3) # [batch, seq, heads, log(seq), head_dim]

        scores = torch.einsum("...ij,...jk->...ik", Q, K.transpose(-1, -2)).squeeze(-1) # single vector attention #[batch, seq, heads, log(seq)]
        normalized_scores = scores / self.scale
        normalized_scores = torch.where(normalized_scores == 0, torch.full_like(normalized_scores, float('-inf')), normalized_scores) # mask attention
        attention_weights = torch.softmax(normalized_scores, dim=-1)

        attention_output = torch.einsum("...ki,...ij->...kj", attention_weights.unsqueeze(-2), V).squeeze(-2) # [batch, seq, heads, head_dim]
        attention_output = attention_output.reshape(batch_size, seq, embedding)
        output = self.out_proj(attention_output)

        return output

    def forward(self, x: Tensor) -> Tensor: # [batch_size, seq_len, embed_dim]
        """
        Forward pass that selects the desired memory processing method.

        This method is a placeholder for selecting a concrete memory processing
        scheme to be applied to input `x`. It's meant to choose from different
        access methods like `parallel`, `sequential`, or `attention`.

        Args:
            x (Tensor): Input tensor of shape (batch_size, seq_len, embedding_dim).

        Returns:
           Tensor: Output tensor of shape (batch_size, seq_len, embedding_dim).
        """
        x_ = self.norm1(x)
        x_ = self.parallel(x_) if self.parallelize else self.sequential(x_)
        x = x + self.attention(x_)
        x_ = self.norm2(x)
        x = x + self.feedforward(x_)
        return x

    def __call__(self, *args, **kwds) -> Tensor:
        return super().__call__(*args, **kwds)

# GPT-2

In [4]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [5]:
# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[LogarithmicMemory(n_embd, n_head, 2) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        # pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb# + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [6]:
model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

0.071489 M parameters
step 0: train loss 4.3501, val loss 4.3431
step 100: train loss 2.5552, val loss 2.5653
step 200: train loss 2.2462, val loss 2.2586
step 300: train loss 2.1105, val loss 2.1414
step 400: train loss 2.0223, val loss 2.0719
step 500: train loss 1.9674, val loss 2.0350
step 600: train loss 1.9227, val loss 1.9965
step 700: train loss 1.8891, val loss 1.9796
step 800: train loss 1.8576, val loss 1.9658
step 900: train loss 1.8348, val loss 1.9562
step 1000: train loss 1.8104, val loss 1.9378
step 1100: train loss 1.7930, val loss 1.9301
step 1200: train loss 1.7712, val loss 1.9189
step 1300: train loss 1.7595, val loss 1.9082
step 1400: train loss 1.7460, val loss 1.9040
step 1500: train loss 1.7395, val loss 1.9011
step 1600: train loss 1.7284, val loss 1.8916
step 1700: train loss 1.7153, val loss 1.8754
step 1800: train loss 1.7074, val loss 1.8783
step 1900: train loss 1.6999, val loss 1.8682
step 2000: train loss 1.6910, val loss 1.8566
step 2100: train loss 1.

# End